# 02 — Track Kinematics

Compute translation speed between consecutive best-track fixes using `src.track_utils`.
The illustrative 6-hourly subset mirrors IBTrACS v4 layout — swap in the official CSV for analysis.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from src.track_utils import haversine_km, translation_speed_kmh

track = pd.read_csv('../data/biparjoy_besttrack_sample.csv', parse_dates=['time_utc'])
track = track.sort_values('time_utc').reset_index(drop=True)
track.head(3)


In [ ]:
# Step-wise distance and translation speed between consecutive fixes
rows = []
for i in range(1, len(track)):
    a, b = track.iloc[i - 1], track.iloc[i]
    d = haversine_km(a.lon, a.lat, b.lon, b.lat)
    dt = (b.time_utc - a.time_utc).total_seconds() / 3600.0
    rows.append({
        'time_utc': b.time_utc,
        'step_km': round(d, 1),
        'dt_h': dt,
        'speed_kmh': round(translation_speed_kmh(d, dt) or 0, 1),
    })
kin = pd.DataFrame(rows)
kin.tail()


In [ ]:
# Slow translation near landfall is a hallmark of Biparjoy — visualise it
import matplotlib.pyplot as plt
from src.visualization import apply_publication_style

apply_publication_style()
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(kin.time_utc, kin.speed_kmh, marker='o')
ax.set_ylabel('Translation speed (km h⁻¹)')
ax.set_title('Biparjoy translation speed (6-hourly fixes)')
plt.show()
